# Chimanimani Stage 3 and 4: anomaly detection, then validation against the published Idai inventory
Cells 1-6 are the Stage 3 cells that already ran (the notebook now also searches Google Drive for the MintPy outputs). Cells 7-11 are new.
If the Stage 3 session is still open, only cells 7-11 need to be run in that same session.

7. Download the Dille et al. Idai inventory from UNESCO IHP-WINS (DOI 10.63253/nii4g2ac), verify it, load it, quality-check it
8. Primary validation: terrain-stratified Mantel-Haenszel odds ratio, block-bootstrap interval, shift test
9. Sensitivity of the result to the detector settings
10. Logistic regression with repeated spatial block cross-validation, against the published susceptibility map
11. Save outputs


In [ ]:
# Google Colab: download the current notebook (.ipynb) to your phone
import json
import datetime
from google.colab import files, _message

# Get the current notebook as JSON
nb_json = _message.blocking_request('get_ipynb', timeout_sec=10)['ipynb']

# Make a filename with date/time
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Colab_Notebook_{stamp}.ipynb"

# Save it inside the Colab VM
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(nb_json, f, ensure_ascii=False, indent=1)

# Trigger download to your device
files.download(filename)

In [ ]:

# ===== PREPEND TO CELL 7: imports + rebuild Stage-3 state if the kernel is fresh =====
import os, sys, glob, shutil, subprocess, zipfile, time
import numpy as np, pandas as pd, h5py
from scipy import ndimage
import matplotlib.pyplot as plt

def _ensure(pkg):
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
for _p in ('geopandas', 'rasterio', 'scikit-learn'): _ensure(_p)

import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from google.colab import files
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

if 'A' not in globals() or 'mask_B' not in globals() or 'hs' not in globals():

    MINTPY_FOLDER  = 'chimanimani_mintpy_run1'
    DRIVE_ROOT     = '/content/drive/MyDrive'
    MINTPY_DIR     = os.path.join(DRIVE_ROOT, MINTPY_FOLDER)
    OUT_DIR        = '/content/data/stage3'
    MIN_COH        = 0.7
    K_VEL          = 2.5
    BG_BLOCK_PX    = 50
    MIN_PIXELS     = 20
    SLOPE_MIN_DEG  = 5.0
    K_TS           = 2.5
    MIN_RUN        = 3
    MIN_PIXELS_TS  = 10
    BUFFER_PX      = 3
    N_PERM         = 1000
    BLOCK_CV_PX    = 50
    SEED           = 42

    def ok(m):   print(f'[PASS] {m}')
    def warn(m): print(f'[WARN] {m}')
    def skip(m): print(f'[SKIP] {m}')
    def fail(m):
        print(f'[FAIL] {m}')
        raise RuntimeError('STOPPED: ' + m)

    os.makedirs(OUT_DIR, exist_ok=True)
    rng = np.random.default_rng(SEED)

    REQ_FILES = ['velocity.h5', 'temporalCoherence.h5', 'timeseries_tropHgt_demErr.h5', 'inputs/geometryGeo.h5']
    def is_run(d): return all(os.path.exists(os.path.join(d, f)) for f in REQ_FILES)

    def mount_drive():
        if os.path.isdir(DRIVE_ROOT): return True
        try:
            from google.colab import drive
            drive.mount(os.path.dirname(DRIVE_ROOT))
        except Exception as e:
            warn(f'Could not mount Google Drive ({type(e).__name__}: {e})')
        return os.path.isdir(DRIVE_ROOT)

    def walk_drive(match, max_depth=8, max_seconds=240):
        t0, hits = time.time(), []
        base_depth = DRIVE_ROOT.rstrip('/').count('/')
        for dp, dn, fn in os.walk(DRIVE_ROOT):
            dn[:] = [d for d in dn if not d.startswith('.')]
            if dp.count('/') - base_depth >= max_depth: dn[:] = []
            if match(dp, fn): hits.append((dp, fn))
            if time.time() - t0 > max_seconds:
                warn(f'Drive search stopped after {max_seconds} s; results may be incomplete'); break
        return hits

    if is_run(MINTPY_DIR):
        ok(f'MintPy outputs found at {MINTPY_DIR}')
    else:
        print(f"'{MINTPY_DIR}' not ready; mounting Drive and searching for '{MINTPY_FOLDER}' ...")
        if not mount_drive(): fail('Google Drive is not available and the local MintPy folder is missing')
        hits = [dp for dp, _ in walk_drive(lambda dp, fn: os.path.basename(dp) == MINTPY_FOLDER and is_run(dp))]
        if not hits:
            fail(
                f"Could not find folder '{MINTPY_FOLDER}' with a complete MintPy run in Google Drive.\n"
                f"  Required files: {', '.join(REQ_FILES)}\n"
                f"  Check that the folder is named exactly '{MINTPY_FOLDER}' in your Drive."
            )
        MINTPY_DIR = hits[0]
        ok(f"Found '{MINTPY_FOLDER}' at: {MINTPY_DIR}")

    def read_h5(path, name):
        if not os.path.exists(path): fail(f'Missing file {path}. Point MINTPY_DIR to the folder holding the MintPy outputs.')
        with h5py.File(path, 'r') as f:
            if name not in f: fail(f"Dataset '{name}' not found in {path}. Found: {list(f.keys())}")
            attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
            return f[name][:], attrs

    vel, vattr = read_h5(f'{MINTPY_DIR}/velocity.h5', 'velocity')
    vstd, _    = read_h5(f'{MINTPY_DIR}/velocity.h5', 'velocityStd')
    tcoh, _    = read_h5(f'{MINTPY_DIR}/temporalCoherence.h5', 'temporalCoherence')
    height, _  = read_h5(f'{MINTPY_DIR}/inputs/geometryGeo.h5', 'height')
    wmask, _   = read_h5(f'{MINTPY_DIR}/inputs/geometryGeo.h5', 'waterMask')
    ts, _      = read_h5(f'{MINTPY_DIR}/timeseries_tropHgt_demErr.h5', 'timeseries')
    dates, _   = read_h5(f'{MINTPY_DIR}/timeseries_tropHgt_demErr.h5', 'date')
    dates = pd.to_datetime(dates.astype(str), format='%Y%m%d')
    t_years = np.asarray((dates - dates[0]).days) / 365.25
    print(f'Loaded velocity {vel.shape}, time series {ts.shape}, {len(dates)} dates ({dates[0].date()} to {dates[-1].date()})')

    shapes = {'velocity': vel.shape, 'velocityStd': vstd.shape, 'temporalCoherence': tcoh.shape, 'height': height.shape, 'waterMask': wmask.shape, 'timeseries': ts.shape[1:]}
    if len(set(shapes.values())) != 1: fail(f'Grid sizes differ between files: {shapes}')
    ok(f'All layers share one grid {vel.shape}')
    if ts.shape[0] != len(dates): fail(f'Time series has {ts.shape[0]} layers but {len(dates)} dates')
    ok(f'Time series has {len(dates)} dates')

    x_step, y_step = float(vattr['X_STEP']), float(vattr['Y_STEP'])
    x0, y0 = float(vattr['X_FIRST']), float(vattr['Y_FIRST'])
    if abs(x_step) < 1: fail(f'Pixel size {x_step} looks like degrees; this notebook needs a projected grid in metres')
    ok(f'Pixel size {abs(x_step):.0f} m x {abs(y_step):.0f} m')
    if np.nanmedian(np.abs(vel)) > 1: fail('Median |velocity| above 1 m/yr: units look wrong')
    ok('Velocity units look like metres per year')

    vel_mm, vstd_mm, ts_mm = vel * 1000.0, vstd * 1000.0, ts * 1000.0
    water = ~wmask.astype(bool)
    if water.mean() > 0.5: fail('waterMask marks more than half the grid as water: mask looks inverted')

    n_all = vel.size
    f1 = np.isfinite(vel_mm) & np.isfinite(height)
    f2 = f1 & np.isfinite(tcoh) & (tcoh >= MIN_COH)
    valid = f2 & ~water
    print(f'Pixels: {n_all} total -> {f1.sum()} finite -> {f2.sum()} with coherence >= {MIN_COH} -> {valid.sum()} on land ({valid.mean():.0%})')
    if valid.mean() < 0.2: fail('Fewer than 20% of pixels are usable')
    ok('Enough usable pixels')

    def mad_sigma(x):
        x = x[np.isfinite(x)]
        return 1.4826 * np.median(np.abs(x - np.median(x)))

    def terrain(h, x_step, y_step):
        dzdx = np.gradient(h, axis=1) / x_step
        dzdy = np.gradient(h, axis=0) / y_step
        slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
        aspect = np.degrees(np.arctan2(-dzdx, -dzdy)) % 360
        curv = np.gradient(dzdx, axis=1) / x_step + np.gradient(dzdy, axis=0) / y_step
        return slope, aspect, curv

    def block_background(field, valid, block):
        ny, nx = field.shape
        by, bx = -(-ny // block), -(-nx // block)
        coarse = np.full((by, bx), np.nan)
        for i in range(by):
            for j in range(bx):
                sl = (slice(i * block, (i + 1) * block), slice(j * block, (j + 1) * block))
                v = field[sl][valid[sl]]
                if v.size >= max(10, 0.05 * block * block): coarse[i, j] = np.median(v)
        if np.isnan(coarse).all(): raise ValueError("No block has enough valid pixels")
        idx = ndimage.distance_transform_edt(np.isnan(coarse), return_distances=False, return_indices=True)
        coarse = ndimage.gaussian_filter(coarse[tuple(idx)], 1.0, mode='nearest')
        yy = (np.arange(ny) + 0.5) / block - 0.5
        xx = (np.arange(nx) + 0.5) / block - 0.5
        return ndimage.map_coordinates(coarse, np.array(np.meshgrid(yy, xx, indexing='ij')), order=1, mode='nearest')

    def detect_velocity_clusters(vel_mm, valid, slope, k, block, min_pixels, slope_min):
        bg = block_background(vel_mm, valid, block)
        resid = np.where(valid, vel_mm - bg, np.nan)
        sigma = mad_sigma(resid[valid])
        z = resid / sigma
        steps = {'valid pixels': int(valid.sum())}
        labels = np.zeros(vel_mm.shape, dtype=np.int32)
        sign = {}
        nxt = 0
        s8 = np.ones((3, 3), dtype=int)
        n_cand = n_raw = n_size = 0
        for s in (1, -1):
            cand = valid & (s * z > k)
            n_cand += int(cand.sum())
            lab, n = ndimage.label(cand, structure=s8)
            n_raw += n
            if n == 0: continue
            area = np.bincount(lab.ravel(), minlength=n + 1)
            mean_slope = ndimage.mean(slope, lab, index=np.arange(1, n + 1))
            for i in range(1, n + 1):
                if area[i] < min_pixels: continue
                n_size += 1
                if mean_slope[i - 1] < slope_min: continue
                nxt += 1
                labels[lab == i] = nxt
                sign[nxt] = s
        steps.update({f'pixels with |z| > {k}': n_cand, 'raw clusters': n_raw,
                      f'clusters >= {min_pixels} px': n_size, f'clusters on slope >= {slope_min} deg': nxt})
        return dict(labels=labels, sign=sign, z=z, resid=resid, bg=bg, sigma=sigma, steps=steps)

    def cluster_table(labels, sign, vel_mm, z, slope, tcoh, x0, y0, xs, ys, to_lonlat=None, pixel_area_m2=None):
        ids = [i for i in np.unique(labels) if i > 0]
        pix_area = abs(xs * ys) if pixel_area_m2 is None else pixel_area_m2
        rows = []
        for i in ids:
            m = labels == i
            r, c = ndimage.center_of_mass(m)
            x, y = x0 + (c + 0.5) * xs, y0 + (r + 0.5) * ys
            row = dict(cluster_id=int(i), sign=int(sign.get(i, 0)), n_pixels=int(m.sum()), area_ha=m.sum() * pix_area / 1e4,
                       mean_resid_mm_yr=float(np.nanmean(vel_mm[m])), max_abs_z=float(np.nanmax(np.abs(z[m]))),
                       mean_slope_deg=float(np.nanmean(slope[m])), mean_coherence=float(np.nanmean(tcoh[m])), x=x, y=y)
            if to_lonlat is not None: row['lon'], row['lat'] = to_lonlat(x, y)
            rows.append(row)
        return pd.DataFrame(rows)

    def persistent_flags(flag_signed, min_run):
        n = flag_signed.shape[0]
        out = np.zeros(flag_signed.shape, dtype=bool)
        for s in (1, -1):
            f = flag_signed == s
            fw = np.zeros(f.shape, dtype=np.int32); bw = np.zeros(f.shape, dtype=np.int32)
            for i in range(n):
                fw[i] = (fw[i - 1] + 1) * f[i] if i else f[i]
            for i in range(n - 1, -1, -1):
                bw[i] = (bw[i + 1] + 1) * f[i] if i < n - 1 else f[i]
            out |= f & (fw + bw - 1 >= min_run)
        return out

    def detect_ts_events(ts_mm, t_years, valid, k, min_run, rng=None, n_null=1):
        n, ny, nx = ts_mm.shape
        Y = ts_mm.reshape(n, -1)
        use = valid.ravel() & np.isfinite(Y).all(axis=0)
        tc = t_years - t_years.mean()
        Yu = Y[:, use]
        def wfit(W):
            S0, S1, S2 = W.sum(0), (W * tc[:, None]).sum(0), (W * tc[:, None] ** 2).sum(0)
            Sy, Sty = (W * Yu).sum(0), (W * tc[:, None] * Yu).sum(0)
            det = S0 * S2 - S1 ** 2
            b = (S0 * Sty - S1 * Sy) / det
            a = (Sy - b * S1) / S0
            return Yu - (a + b * tc[:, None])
        resid = wfit(np.ones_like(Yu))
        for _ in range(2):
            sig_r = 1.4826 * np.median(np.abs(resid - np.median(resid, axis=0)), axis=0)
            resid = wfit((np.abs(resid) <= 2.5 * np.maximum(sig_r, 1e-6)).astype(float))
        sig = 1.4826 * np.median(np.abs(resid - np.median(resid, axis=0)), axis=0)
        thr = k * np.maximum(sig, np.median(sig))
        def count(res):
            sgn = np.where(np.abs(res) > thr, np.sign(res), 0)
            return persistent_flags(sgn, min_run).sum(axis=0)
        n_real = count(resid)
        null_rates = []
        if rng is not None:
            for _ in range(n_null):
                null_rates.append(float((count(resid[rng.permutation(n)]) > 0).mean()))
        out_n = np.zeros(ny * nx, dtype=np.int16); out_n[use] = n_real
        out_s = np.full(ny * nx, np.nan, dtype=np.float32); out_s[use] = sig
        return dict(n_flag_dates=out_n.reshape(ny, nx), sigma=out_s.reshape(ny, nx), thr_floor=float(k * np.median(sig)),
                    real_rate=float((n_real > 0).mean()), null_rate=float(np.mean(null_rates)) if null_rates else None)

    def label_filter(mask, min_pixels):
        lab, n = ndimage.label(mask, structure=np.ones((3, 3), dtype=int))
        if n == 0: return np.zeros(mask.shape, dtype=np.int32), 0
        area = np.bincount(lab.ravel(), minlength=n + 1)
        keep = np.zeros(n + 1, dtype=np.int32)
        ids = [i for i in range(1, n + 1) if area[i] >= min_pixels]
        keep[ids] = np.arange(1, len(ids) + 1)
        return keep[lab], len(ids)

    slope, aspect, curv = terrain(np.where(np.isfinite(height), height, np.nanmedian(height)), x_step, y_step)
    print(f"Slope: median {np.median(slope[valid]):.1f} deg, 95th percentile {np.percentile(slope[valid], 95):.1f} deg")
    if np.median(slope[valid]) < 1 or np.median(slope[valid]) > 60: fail("Slope values are implausible for this terrain")
    ok("Terrain derivatives computed")

    A = detect_velocity_clusters(vel_mm, valid, slope, K_VEL, BG_BLOCK_PX, MIN_PIXELS, SLOPE_MIN_DEG)
    print(f"\nNoise level after background removal: sigma = {A['sigma']:.1f} mm/yr (robust)")
    if A['sigma'] > 20: warn("Noise level above 20 mm/yr: only very fast motion can stand out")
    for k_, v_ in A['steps'].items(): print(f"   {k_}: {v_}")
    n_clusters = len(A['sign'])
    (ok if n_clusters > 0 else warn)(f"{n_clusters} velocity clusters kept")

    try:
        from pyproj import Transformer
        epsg = None
        for p in glob.glob('/content/data/aoi/*/*_dem.tif'):
            from osgeo import gdal, osr
            srs = osr.SpatialReference(wkt=gdal.Open(p).GetProjection())
            epsg = int(srs.GetAuthorityCode(None)); break
        if epsg is None:
            epsg = int(vattr.get('EPSG', 32736)); warn(f"Cropped GeoTIFFs not found; assuming EPSG:{epsg}")
        _tr = Transformer.from_crs(epsg, 4326, always_xy=True)
        to_lonlat = lambda x, y: _tr.transform(x, y)
        ok(f"Coordinates in EPSG:{epsg}")
    except Exception as e:
        to_lonlat, epsg = None, None
        warn(f"No lon/lat conversion ({type(e).__name__}); table will hold map coordinates only")

    tab_A = cluster_table(A['labels'], A['sign'], A['resid'], A['z'], slope, tcoh, x0, y0, x_step, y_step, to_lonlat)
    if len(tab_A):
        print("\nLargest clusters:")
        print(tab_A.sort_values('n_pixels', ascending=False).head(10).round(2).to_string(index=False))

    B = detect_ts_events(ts_mm, t_years, valid, K_TS, MIN_RUN, rng=rng, n_null=5)
    print(f"Residual scatter: median {np.nanmedian(B['sigma']):.1f} mm; floor for the k-sigma threshold {B['thr_floor']:.1f} mm")
    print(f"Pixels with a persistent anomaly: {B['real_rate']:.2%} real, {B['null_rate']:.2%} in shuffled-date null")
    if B['real_rate'] <= 1.5 * max(B['null_rate'], 1e-6): warn("Time-series flags are not clearly above the shuffled-date null")
    else: ok("Time-series flags exceed the shuffled-date null")

    lab_B, n_B = label_filter(B['n_flag_dates'] >= MIN_RUN, MIN_PIXELS_TS)
    print(f"Time-series anomaly clusters (>= {MIN_PIXELS_TS} px): {n_B}")
    mask_B = lab_B > 0
    if len(tab_A):
        tab_A['frac_ts_flagged'] = [float(mask_B[A['labels'] == i].mean()) for i in tab_A['cluster_id']]
        tab_A['priority'] = np.where(tab_A['frac_ts_flagged'] > 0.1, 'both', 'velocity only')
        print(tab_A['priority'].value_counts().to_string())
    tab_B = cluster_table(lab_B, {i: 0 for i in range(1, n_B + 1)}, vel_mm, A['z'], slope, tcoh, x0, y0, x_step, y_step, to_lonlat) if n_B else pd.DataFrame()

    tab_A.to_csv(f'{OUT_DIR}/velocity_clusters.csv', index=False)
    tab_B.to_csv(f'{OUT_DIR}/timeseries_clusters.csv', index=False)

    layers = {'velocity_resid_mm_yr': A['resid'], 'z_score': A['z'], 'velocity_clusters': A['labels'].astype(np.float32),
              'ts_anomaly_dates': B['n_flag_dates'].astype(np.float32), 'slope_deg': slope}
    try:
        from osgeo import gdal, osr
        srs = osr.SpatialReference(); srs.ImportFromEPSG(epsg)
        for name, arr in layers.items():
            ds = gdal.GetDriverByName('GTiff').Create(f'{OUT_DIR}/{name}.tif', arr.shape[1], arr.shape[0], 1, gdal.GDT_Float32)
            ds.SetGeoTransform((x0, x_step, 0, y0, 0, y_step)); ds.SetProjection(srs.ExportToWkt())
            ds.GetRasterBand(1).WriteArray(np.nan_to_num(arr.astype(np.float32), nan=-9999)); ds.GetRasterBand(1).SetNoDataValue(-9999)
            ds = None
        ok(f"Wrote {len(layers)} GeoTIFFs")
    except Exception as e:
        np.savez_compressed(f'{OUT_DIR}/layers.npz', **layers)
        warn(f"GeoTIFF export failed ({type(e).__name__}); saved layers.npz instead")

    from matplotlib.colors import LightSource
    ls = LightSource(azdeg=315, altdeg=45)
    hs = ls.hillshade(np.where(np.isfinite(height), height, np.nanmedian(height)), vert_exag=2, dx=abs(x_step), dy=abs(y_step))
    fig, ax = plt.subplots(1, 3, figsize=(20, 6))
    for a, (t, d, vm) in zip(ax[:2], [('Velocity minus background (mm/yr)', A['resid'], 3 * A['sigma']), ('z-score', A['z'], 4)]):
        a.imshow(hs, cmap='gray'); im = a.imshow(d, cmap='RdBu_r', vmin=-vm, vmax=vm, alpha=0.7); a.set_title(t); plt.colorbar(im, ax=a, shrink=0.7)
    ax[2].imshow(hs, cmap='gray')
    ax[2].contour(A['labels'] > 0, levels=[0.5], colors='red', linewidths=0.8)
    ax[2].contour(mask_B, levels=[0.5], colors='cyan', linewidths=0.8)
    ax[2].set_title('Velocity clusters (red), time-series anomalies (cyan)')
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/stage3_overview.png', dpi=130); plt.show()
    ok(f"Outputs in {OUT_DIR}: {sorted(os.listdir(OUT_DIR))}")

    if mount_drive():
        shutil.copytree(OUT_DIR, os.path.join(DRIVE_ROOT, 'chimanimani_stage3'), dirs_exist_ok=True); ok("Copied to Google Drive")
    else:
        warn("Google Drive unavailable: outputs stay on the Colab disk only")

else:
    print("[SKIP] Stage-3 state already in memory — Cell 7 will use it as-is.")

# ===== CELL 7: upload inventory files, verify exact filenames, quality-check =====
INV_DIR             = '/content/data/idai_inventory'
EXTENT_CLASS_FIELD  = 'Zone'
SOURCE_CLASS_VALUES = ['Depletion zone']
INV_BUFFER_M        = 100
ELIGIBLE_SLOPE_MIN  = SLOPE_MIN_DEG

EXPECTED = {
    'points': 'idai-landslide-source-points.zip',
    'extent': 'idai-landslide-extent-maps_zones.zip',
    'susc':   'landslide-susceptibility-map-for-landslide-sources-in-chimanimani-and-chipinge_cog.tif',
}

os.makedirs(INV_DIR, exist_ok=True)

def upload_exact(key, description):
    expected = EXPECTED[key]
    print()
    print('=' * 72)
    print(f'  UPLOAD REQUIRED  —  {description}')
    print(f'  Expected filename (EXACT, case-sensitive):')
    print(f'    {expected}')
    print('=' * 72)
    uploaded = files.upload()
    if len(uploaded) == 0:
        fail(f"No file was uploaded for '{key}'. Pipeline stopped.")
    name = list(uploaded.keys())[0]
    if name != expected:
        fail(
            f'Filename mismatch — pipeline stopped.\n'
            f"  Expected : '{expected}'\n"
            f"  Uploaded : '{name}'\n"
            f'  Rename the file to match exactly, then re-run this cell.'
        )
    dest = os.path.join(INV_DIR, name)
    with open(dest, 'wb') as f:
        f.write(uploaded[name])
    if name.endswith('.zip'):
        with zipfile.ZipFile(dest) as z:
            bad = z.testzip()
            if bad: fail(f'Uploaded zip is corrupt (bad member: {bad}). Re-upload a valid copy.')
    ok(f'Accepted: {name}  ({os.path.getsize(dest)/1e6:.1f} MB)')
    return dest

def read_vector(zip_path):
    d = os.path.join(INV_DIR, os.path.basename(zip_path)[:-4])
    if not os.path.isdir(d):
        with zipfile.ZipFile(zip_path) as z: z.extractall(d)
    shps = glob.glob(f'{d}/**/*.shp', recursive=True)
    if len(shps) != 1: fail(f'Expected exactly one .shp in {zip_path}, found {len(shps)}: {shps}')
    g = gpd.read_file(shps[0])
    if g.crs is None: fail(f'{shps[0]} has no coordinate system (.prj missing)')
    return g[g.geometry.notna() & ~g.geometry.is_empty].copy()

def read_susc(path):
    with rasterio.open(path) as src:
        nod = src.nodata
        with WarpedVRT(src, crs=f'EPSG:{epsg}', transform=transform, width=nx_g, height=ny_g,
                       resampling=Resampling.bilinear, nodata=nod) as vrt:
            a = vrt.read(1).astype('float32')
    if nod is not None: a[a == nod] = np.nan
    return a

if epsg is None: fail('Grid EPSG code unknown; cannot place the inventory on the grid')
ny_g, nx_g = vel.shape
transform = Affine(x_step, 0, x0, 0, y_step, y0)

pts_path = upload_exact('points', 'Landslide SOURCE POINTS shapefile (zip)')
ext_path = upload_exact('extent', 'Landslide EXTENT MAPS shapefile (zip)  [contains Depletion zone scars]')
susc_path = upload_exact('susc',   'Published susceptibility GeoTIFF (landslide sources COG)')

G_pts = read_vector(pts_path)
G_ext = read_vector(ext_path)
print(f'Source points: {len(G_pts)} features, CRS {G_pts.crs.to_string()}, geometry {sorted(G_pts.geom_type.unique())}')
print(f'Extent maps:   {len(G_ext)} features, CRS {G_ext.crs.to_string()}, fields {[c for c in G_ext.columns if c != "geometry"]}')
G_pts, G_ext = G_pts.to_crs(epsg), G_ext.to_crs(epsg)
if not (G_pts.geom_type == 'Point').all():
    warn('Non-point geometries in the source points; using centroids'); G_pts['geometry'] = G_pts.geometry.centroid
px, py = G_pts.geometry.x.to_numpy(), G_pts.geometry.y.to_numpy()

rows_all = np.floor((py - y0) / y_step).astype(int); cols_all = np.floor((px - x0) / x_step).astype(int)
inside = (rows_all >= 0) & (rows_all < ny_g) & (cols_all >= 0) & (cols_all < nx_g)
pr, pc = rows_all[inside], cols_all[inside]
elig = valid & (slope >= ELIGIBLE_SLOPE_MIN)
dups = int(pd.Series(list(zip(np.round(px[inside], 1), np.round(py[inside], 1)))).duplicated().sum())
print(f'\nPoints inside the AOI grid: {inside.sum()} of {len(px)} ({dups} exact duplicates)')
print(f'   on usable InSAR pixels:                {int(valid[pr, pc].sum())} ({valid[pr, pc].mean():.0%})')
print(f'   on usable pixels with slope >= {ELIGIBLE_SLOPE_MIN:.0f} deg: {int(elig[pr, pc].sum())} ({elig[pr, pc].mean():.0%})')
if inside.sum() < 100: fail('Fewer than 100 inventory points inside the AOI: check the CRS and the grid')
area_km2 = ny_g * nx_g * abs(x_step * y_step) / 1e6
print(f'   density inside the AOI: {inside.sum() / area_km2:.1f} landslide points per km2 (AOI {area_km2:.0f} km2)')
L = np.zeros(vel.shape, dtype=bool); L[pr, pc] = True
n_L = int((L & elig).sum())
(ok if n_L >= 100 else warn)(f'{n_L} eligible landslide pixels (usable InSAR, slope >= {ELIGIBLE_SLOPE_MIN:.0f} deg)')
if n_L < 30: fail('Fewer than 30 eligible landslide pixels: the InSAR data cannot validate against this inventory')

fld, vals = EXTENT_CLASS_FIELD, SOURCE_CLASS_VALUES
src_polys = G_ext[G_ext[fld].isin(vals)]
if len(src_polys) == 0: fail(f'No polygons with {fld} in {vals} — check the correct shapefile was uploaded')
ok(f'Landslide-source (Depletion zone) polygons: {len(src_polys)}')
src_mask = features.rasterize(
    ((g, 1) for g in src_polys.geometry), out_shape=vel.shape,
    transform=transform, fill=0, all_touched=True, dtype='uint8'
).astype(bool)
buf_px = int(np.ceil(INV_BUFFER_M / abs(x_step)))
src_near = ndimage.binary_dilation(src_mask, structure=np.ones((3, 3), dtype=bool), iterations=buf_px)
confirmed = src_near[pr, pc]
L_conf = np.zeros(vel.shape, dtype=bool); L_conf[pr[confirmed], pc[confirmed]] = True
print(f'   points within {INV_BUFFER_M} m of a mapped source polygon: {confirmed.sum()} of {len(pr)} ({confirmed.mean():.0%}); the paper reports 73.8% over both districts')
print(f'   eligible event-confirmed landslide pixels: {int((L_conf & elig).sum())}')

susc = read_susc(susc_path)
cover = float(np.isfinite(susc[valid]).mean())
print(f'Published susceptibility map: finite on {cover:.0%} of usable pixels; values {np.nanpercentile(susc[valid], [1, 50, 99]).round(3)} (1st/50th/99th percentile)')
if cover < 0.9: fail('The susceptibility map does not cover the AOI')
ok('All three inventory files loaded and validated — ready for Cells 8-11')

'/content/drive/MyDrive/chimanimani_mintpy_run1' not ready; mounting Drive and searching for 'chimanimani_mintpy_run1' ...
[WARN] Could not mount Google Drive (MessageError: Error: credential propagation was unsuccessful)
[FAIL] Google Drive is not available and the local MintPy folder is missing


RuntimeError: STOPPED: Google Drive is not available and the local MintPy folder is missing

In [ ]:
# ===== CELL 8: primary validation: Mantel-Haenszel association with block bootstrap and shift test =====
def make_strata(slope, height, elig):
    s_bin = np.clip((slope // 2).astype(int), 0, 29)
    e_bin = np.clip(((height - np.nanmin(height[elig])) // 100).astype(int), 0, None)
    uniq, inv = np.unique((s_bin * 1000 + e_bin)[elig], return_inverse=True)
    out = np.full(slope.shape, -1, dtype=np.int64); out[elig] = inv
    return out, len(uniq)

def mh_or(tot):
    a, b, c, d = tot[:, 0], tot[:, 1], tot[:, 2], tot[:, 3]
    n = a + b + c + d
    ok_ = n > 0
    num, den = np.sum(a[ok_] * d[ok_] / n[ok_]), np.sum(b[ok_] * c[ok_] / n[ok_])
    return num / den if den > 0 else np.nan

def cat_of(F, Lm):
    return np.where(F & Lm, 0, np.where(F & ~Lm, 1, np.where(~F & Lm, 2, 3)))

strat, S = make_strata(slope, height, elig)
blk = (np.arange(vel.shape[0])[:, None] // BLOCK_CV_PX) * 1000 + (np.arange(vel.shape[1])[None, :] // BLOCK_CV_PX)
ub, blk_idx = np.unique(blk[elig], return_inverse=True)
NB = len(ub)
blk_map = np.full(vel.shape, -1, dtype=np.int64); blk_map[elig] = blk_idx
print(f"Eligible pixels: {int(elig.sum())} in {S} slope x elevation strata and {NB} spatial blocks of {BLOCK_CV_PX * abs(x_step) / 1000:.0f} km")
if NB < 50: warn(f"Only {NB} spatial blocks: confidence intervals will be wide")
n_cl = int((np.unique(A['labels']) > 0).sum())
if n_cl < 10: warn(f"Only {n_cl} velocity clusters: the association rests on very few independent features")

def block_counts(F, Lm):
    counts = np.zeros((NB, S, 4), dtype=np.int64)
    np.add.at(counts, (blk_map[elig], strat[elig], cat_of(F, Lm)[elig]), 1)
    return counts

def mh_with_ci(F, Lm, n_boot=1000):
    counts = block_counts(F, Lm)
    point = mh_or(counts.sum(0))
    reps = np.array([mh_or(counts[rng.integers(0, NB, NB)].sum(0)) for _ in range(n_boot)])
    lo, hi = (np.nanpercentile(reps, [2.5, 97.5]) if np.isfinite(reps).sum() > 20 else (np.nan, np.nan))
    return point, lo, hi

def shift_p(F, Lm, n_shift=500, min_shift=25):
    flat = lambda Fx: np.bincount(strat[elig] * 4 + cat_of(Fx, Lm)[elig], minlength=S * 4).reshape(S, 4)
    obs = mh_or(flat(F)); null = []; tries = 0
    while len(null) < n_shift:
        tries += 1
        if tries > 10 * n_shift: fail("Shift test could not produce enough valid shifted maps")
        dy, dx = rng.integers(-vel.shape[0] // 2, vel.shape[0] // 2), rng.integers(-vel.shape[1] // 2, vel.shape[1] // 2)
        if max(abs(dy), abs(dx)) < min_shift: continue
        v = mh_or(flat(np.roll(F, (dy, dx), axis=(0, 1))))
        if np.isfinite(v): null.append(v)
    return obs, (1 + np.sum(np.array(null) >= obs)) / (1 + len(null)), float(np.median(null))

def dilate(mask, n):
    return ndimage.binary_dilation(mask, structure=np.ones((3, 3), dtype=bool), iterations=n) if n > 0 else mask

F_vel = dilate(A['labels'] > 0, BUFFER_PX)
RESULTS = []
def report(name, F, Lm):
    n_F = int((F & elig).sum())
    p_F_L = float(F[Lm & elig].mean()); p_F_nL = float(F[~Lm & elig].mean())
    est, lo, hi = mh_with_ci(F, Lm)
    obs, p, med = shift_p(F, Lm)
    print(f"\n{name}")
    print(f"   flagged share of eligible pixels: {n_F / elig.sum():.2%}")
    print(f"   landslide pixels flagged: {p_F_L:.2%}   other eligible pixels flagged: {p_F_nL:.2%}   crude ratio {p_F_L / max(p_F_nL, 1e-9):.2f}")
    print(f"   terrain-stratified (Mantel-Haenszel) odds ratio {est:.2f}, 95% block-bootstrap CI [{lo:.2f}, {hi:.2f}]")
    print(f"   shift test: observed {obs:.2f} vs median of shifted maps {med:.2f}, one-sided p = {p:.3f}")
    verdict = 'ASSOCIATION' if (lo > 1 and p < 0.05) else 'NO CLEAR ASSOCIATION'
    (ok if verdict == 'ASSOCIATION' else warn)(f"{verdict} between flagged clusters and 2019 landslide sources")
    RESULTS.append(dict(analysis=name, flagged_share=n_F / elig.sum(), crude_ratio=p_F_L / max(p_F_nL, 1e-9), mh_or=est, ci_lo=lo, ci_hi=hi, shift_p=p, verdict=verdict))

report(f"PRIMARY: velocity clusters (K={K_VEL}, buffer {BUFFER_PX} px) vs all source points", F_vel, L)
report(f"Same flags vs event-confirmed points only", F_vel, L_conf)

In [ ]:
# ===== CELL 9: sensitivity of the association to the detector settings (not tuned on the outcome) =====
rows_ = []
for K in (2.0, 2.5, 3.0):
    Ak = detect_velocity_clusters(vel_mm, valid, slope, K, BG_BLOCK_PX, MIN_PIXELS, SLOPE_MIN_DEG)
    for b in (0, 1, 3):
        Fk = dilate(Ak['labels'] > 0, b)
        for lname, Lm in (('all points', L), ('confirmed points', L_conf)):
            est, lo, hi = mh_with_ci(Fk, Lm, n_boot=500)
            rows_.append(dict(K=K, buffer_px=b, points=lname, clusters=len(Ak['sign']), flagged_share=float((Fk & elig).sum() / elig.sum()), mh_or=est, ci_lo=lo, ci_hi=hi))
for nm, Fx in (('time-series anomalies', mask_B), ('either detector', F_vel | dilate(mask_B, BUFFER_PX))):
    est, lo, hi = mh_with_ci(Fx, L, n_boot=500)
    rows_.append(dict(K=np.nan, buffer_px=BUFFER_PX, points=f'all points ({nm})', clusters=np.nan, flagged_share=float((Fx & elig).sum() / elig.sum()), mh_or=est, ci_lo=lo, ci_hi=hi))
SENS = pd.DataFrame(rows_)
print(SENS.round(3).to_string(index=False))
share_sig = float((SENS['ci_lo'] > 1).mean())
print(f"\nSettings with a lower confidence limit above 1: {share_sig:.0%} of {len(SENS)}")
(ok if share_sig >= 0.75 else warn)("Result is stable across settings" if share_sig >= 0.75 else "Result depends on the settings: report the full table, not one row")

In [ ]:
# ===== CELL 10: logistic regression with repeated spatial block cross-validation =====
feat = {'elevation': height, 'slope': slope, 'sin_aspect': np.sin(np.radians(aspect)), 'cos_aspect': np.cos(np.radians(aspect)),
        'curvature': curv, 'abs_z': np.abs(A['z']), 'in_cluster': (A['labels'] > 0).astype(float), 'published_susc': susc}
MODELS = {'M0 terrain': ['elevation', 'slope', 'sin_aspect', 'cos_aspect', 'curvature'],
          'M1 terrain + InSAR': ['elevation', 'slope', 'sin_aspect', 'cos_aspect', 'curvature', 'abs_z', 'in_cluster'],
          'M2 published susceptibility': ['published_susc'],
          'M3 published susceptibility + InSAR': ['published_susc', 'abs_z', 'in_cluster']}

pos = L & elig
near_pos = dilate(pos, BUFFER_PX)
neg_pool = np.argwhere(elig & ~near_pos)
n_pos = int(pos.sum())
neg = neg_pool[rng.choice(len(neg_pool), size=min(5 * n_pos, len(neg_pool)), replace=False)]
pr_, pc_ = np.nonzero(pos)
R_ = np.concatenate([pr_, neg[:, 0]]); C_ = np.concatenate([pc_, neg[:, 1]])
y = np.concatenate([np.ones(len(pr_)), np.zeros(len(neg))])
ok_rows = np.isfinite(susc[R_, C_]) & np.isfinite(A['z'][R_, C_])
R_, C_, y = R_[ok_rows], C_[ok_rows], y[ok_rows]
groups = (R_ // BLOCK_CV_PX) * 1000 + (C_ // BLOCK_CV_PX)
print(f"Samples: {len(y)} ({int(y.sum())} positive, {int((1 - y).sum())} negative) in {len(np.unique(groups))} spatial blocks; positive share {y.mean():.1%}")
if len(np.unique(groups)) < 10: fail("Fewer than 10 spatial blocks: cannot cross-validate")
X = {m: np.column_stack([feat[c][R_, C_] for c in cols]) for m, cols in MODELS.items()}
for m, Xm in X.items():
    if not np.isfinite(Xm).all(): fail(f"Non-finite values in features of {m}")

N_FOLDS, N_REP = 5, 5
auc = {m: [] for m in MODELS}; ap = {m: [] for m in MODELS}
ug = np.unique(groups)
for rep in range(N_REP):
    fold_of = dict(zip(ug, rng.permutation(len(ug)) % N_FOLDS))
    f = np.array([fold_of[g] for g in groups])
    for k in range(N_FOLDS):
        te, tr = f == k, f != k
        if len(np.unique(y[te])) < 2 or len(np.unique(y[tr])) < 2:
            for m in MODELS: auc[m].append(np.nan); ap[m].append(np.nan)
            continue
        for m in MODELS:
            mdl = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)).fit(X[m][tr], y[tr])
            p = mdl.predict_proba(X[m][te])[:, 1]
            auc[m].append(roc_auc_score(y[te], p)); ap[m].append(average_precision_score(y[te], p))
auc = {m: np.array(v) for m, v in auc.items()}; ap = {m: np.array(v) for m, v in ap.items()}
print(f"\nSpatial block CV ({N_REP} repeats x {N_FOLDS} folds; blocks of {BLOCK_CV_PX * abs(x_step) / 1000:.0f} km):")
LR_ROWS = []
for m in MODELS:
    print(f"   {m:38s} AUC {np.nanmean(auc[m]):.3f} +/- {np.nanstd(auc[m]):.3f}   PR-AUC {np.nanmean(ap[m]):.3f} (chance {y.mean():.3f})")
    LR_ROWS.append(dict(model=m, auc_mean=np.nanmean(auc[m]), auc_sd=np.nanstd(auc[m]), pr_auc_mean=np.nanmean(ap[m])))
for a_, b_, label in (('M1 terrain + InSAR', 'M0 terrain', 'InSAR added to terrain'), ('M3 published susceptibility + InSAR', 'M2 published susceptibility', 'InSAR added to the published susceptibility map')):
    d = auc[a_] - auc[b_]; d = d[np.isfinite(d)]
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"\n{label}: change in AUC {d.mean():+.4f}, 95% range over folds [{lo:+.4f}, {hi:+.4f}]")
    (ok if lo > 0 else warn)("InSAR features improve the spatial cross-validated AUC" if lo > 0 else "No reliable improvement from the InSAR features")
    LR_ROWS.append(dict(model=label, auc_mean=d.mean(), auc_sd=d.std(), pr_auc_mean=np.nan, ci_lo=lo, ci_hi=hi))
mdl = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)).fit(X['M1 terrain + InSAR'], y)
print("\nStandardised coefficients, terrain + InSAR model:")
print(pd.Series(mdl[-1].coef_[0], index=MODELS['M1 terrain + InSAR']).round(3).to_string())

In [ ]:
# ===== CELL 11: save validation outputs =====
res_df = pd.DataFrame(RESULTS)
res_df.to_csv(f'{OUT_DIR}/validation_primary.csv', index=False)
SENS.to_csv(f'{OUT_DIR}/validation_sensitivity.csv', index=False)
pd.DataFrame(LR_ROWS).to_csv(f'{OUT_DIR}/susceptibility_cv.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(hs, cmap='gray')
ax[0].contour(F_vel, levels=[0.5], colors='red', linewidths=0.8)
rr, cc_ = np.nonzero(L & elig)
ax[0].scatter(cc_, rr, s=2, c='yellow', alpha=0.6)
ax[0].set_title('Eligible 2019 landslide pixels (yellow) and velocity clusters (red)')
bins = np.arange(0, 60, 2)
ax[1].hist(slope[elig], bins=bins, density=True, alpha=0.5, label='all eligible pixels')
ax[1].hist(slope[L & elig], bins=bins, density=True, alpha=0.5, label='landslide pixels')
ax[1].set_xlabel('slope (deg)'); ax[1].legend(); ax[1].set_title('Slope distribution')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/validation_overview.png', dpi=130); plt.show()

if mount_drive():
    shutil.copytree(OUT_DIR, os.path.join(DRIVE_ROOT, 'chimanimani_stage3'), dirs_exist_ok=True); ok("Copied to Google Drive")
else:
    warn("Google Drive unavailable: outputs stay on the Colab disk only")
print("\nHow to report this:")
print(" - The inventory is from March 2019 and the InSAR clusters from Oct 2025 - Sep 2026: this tests whether currently moving slopes coincide with 2019 failures, not whether InSAR predicted them.")
print(" - Median 2019 source scar is about 600 m2, smaller than one 40 m pixel (1600 m2): results hold at hillslope scale, not for single scars.")
print(" - Report the odds ratio with its confidence interval, the shift-test p value and the sensitivity table together.")